# Head Composition Analysis

How attention heads compose across layers: QK composition, OV composition,
strongest paths, and activation-level composition strength.

## Why This Matters

Composition head measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.head_composition_analysis import (
    qk_composition_scores, ov_composition_scores,
    strongest_compositions, composition_path_strength,
    head_composition_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## QK Composition

How much do layer 0 heads influence layer 1 queries?

In [ ]:
result = qk_composition_scores(model, source_layer=0, dest_layer=1)
for p in result['per_pair']:
    print(f"  L0.H{p['source_head']} -> L1.H{p['dest_head']}: {p['composition_score']:.4f}")

## OV Composition

In [ ]:
result = ov_composition_scores(model, source_layer=0, dest_layer=1)
for p in result['per_pair'][:8]:  # show first 8
    print(f"  L0.H{p['source_head']} -> L1.H{p['dest_head']}: {p['composition_score']:.4f}")

## Strongest Compositions

In [ ]:
for src in range(3):
    for dst in range(src+1, 4):
        result = strongest_compositions(model, source_layer=src, dest_layer=dst, top_k=2)
        qk_best = result['qk_top'][0]
        ov_best = result['ov_top'][0]
        print(f"  L{src}->L{dst}: QK best=H{qk_best['source_head']}->H{qk_best['dest_head']} "
              f"({qk_best['composition_score']:.4f}), "
              f"OV best=H{ov_best['source_head']}->H{ov_best['dest_head']} "
              f"({ov_best['composition_score']:.4f})")

## Activation-Level Path Strength

In [ ]:
for sh in range(4):
    result = composition_path_strength(model, tokens, source_layer=0, source_head=sh, dest_layer=1, dest_head=0)
    print(f"  L0.H{sh} -> L1.H0: strength={result['composition_strength']:.4f}")

## Summary

In [ ]:
result = head_composition_summary(model, tokens)
for p in result['per_layer_pair']:
    qk = p['strongest_qk']
    ov = p['strongest_ov']
    print(f"  L{p['source_layer']}->L{p['dest_layer']}: "
          f"QK best={qk['composition_score']:.4f}, "
          f"OV best={ov['composition_score']:.4f}")